# Notebook #6: Citation Descriptives
Webegin by generating a summary table, comparing the citation statistics between *sharing* and *non-sharing* articles, as well as a drill-down to the sharing subcategories (granularities). We compare both the raw citation counts and the FWCI scores, providing the median, mean and standard deviation for each group. We also calculate the group-level h-index for each sharing category.

In [1]:
import pandas as pd
from scipy import stats

from helpers import dataset
from helpers.stats import compare_continuous

In [3]:
def calculate_h_index(citations: pd.Series) -> int:
    """Calculates h-index from a Series of citation counts."""
    if citations.empty:
        return 0
    # Sort descending (highest citations first)
    sorted_cits = sorted(citations.dropna().astype(int), reverse=True)
    # Count how many papers have citations >= their rank
    return sum(c >= i + 1 for i, c in enumerate(sorted_cits))

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Summary Table

In [26]:
_IMPACT_AGGS = {
    "TotalCitations": ["count", calculate_h_index, "median", "mean", "std"],
    "FieldWeightedCitationIndex": ["count", "median", "mean", "std"],
}

metrics_per_sharing_class = (
    combined[["data_sharing_class", "TotalCitations", "FieldWeightedCitationIndex"]]
    .groupby("data_sharing_class")
    .agg(_IMPACT_AGGS)
)
metrics_for_any_data_sharing = (
    combined.loc[combined["is_sharing_data"], ["TotalCitations", "FieldWeightedCitationIndex"]]
    .agg(_IMPACT_AGGS)
)

metrics_for_data_sharing = (
    pd.concat([
        metrics_per_sharing_class,
        metrics_for_any_data_sharing.stack().rename("SHARE").reorder_levels([1, 0]).to_frame().T
    ])
    .dropna(how="all", axis=1)
    .drop(columns=[("FieldWeightedCitationIndex", "count")])
    .rename(columns={("TotalCitations", "count"): "N", ("TotalCitations", "calculate_h_index"): "H-Index"})
)
metrics_for_data_sharing

TotalCitations                                                 \
                     count calculate_h_index median       mean        std   
FIXATION              31.0              16.0   18.0  27.032258  22.089943   
NONE                 154.0              32.0   14.0  21.292208  20.260372   
PARTICIPANT           23.0              15.0   20.0  23.478261  24.820940   
TRIAL                 26.0              12.0   11.5  18.076923  17.043293   
SHARE                 80.0              26.0   15.0  23.100000  21.522199   

            FieldWeightedCitationIndex                      
                                median      mean       std  
FIXATION                       1.30210  1.980371  1.746042  
NONE                           0.92025  1.279551  1.183644  
PARTICIPANT                    1.57630  1.675813  1.181175  
TRIAL                          0.78315  1.243019  1.178515  
SHARE                          1.19505  1.653171  1.443364